[<< Sommaire QC](../README.md) | [Version Python](../Python/QC-Py-06b-Derivatives-Valuation-From-Scratch.ipynb) | [Suivant : QC-Py-07-Futures-Forex >>](../Python/QC-Py-07-Futures-Forex.ipynb)

# QC-06b : Valoriser un dérivé — trois moteurs, un seul contrat [Version C#]

**Compétence visée** : valorisation algorithmique d'un produit dérivé (syllabus TimeSpice, certification **RNCP41881 — Expert en finance de marché**).

Jumeau C# du notebook [QC-Py-06b](../Python/QC-Py-06b-Derivatives-Valuation-From-Scratch.ipynb) : **même contrat, mêmes trois moteurs, mêmes verdicts** — écrit en C#/.NET (BCL pur, zéro package NuGet). Le Python montre *comment* pricer dans l'écosystème QuantConnect ; cette version montre que le *même calcul* tient dans le language .NET du cours, avec les mêmes nombres.

Trois moteurs sont confrontés **sur le même produit** : la forme fermée de Black-Scholes, l'arbre binomial, et la simulation Monte-Carlo.

## 1. Le contrat de référence

Un seul contrat traverse tout le carnet — un **call européen** aux paramètres ronds :

| Paramètre | Valeur |
|---|---|
| Spot $S_0$ | 100 |
| Strike $K$ | 100 (à la monnaie) |
| Maturité $T$ | 1 an |
| Taux sans risque $r$ | 5 % (composé continu) |
| Volatilité $\sigma$ | 20 % |
| Dividende | aucun |

Le payoff à maturité est $\max(S_T - K, 0)$. Tout ce qui suit consiste à estimer son espérance actualisée $e^{-rT}\,\mathbb{E}^{\mathbb{Q}}[\max(S_T-K,0)]$ sous la probabilité risque-neutre — par une formule, par un arbre, ou par un échantillon de trajectoires.

In [1]:
using System;
using System.Globalization;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

// Contrat de reference : call europeen ATMF 1 an
double S0 = 100.0, K = 100.0, T = 1.0, R = 0.05, SIGMA = 0.20;
Console.WriteLine($"Contrat : call europeen  S0={S0:0.##}  K={K:0.##}  T={T:0.##} an  r={R:0.##}  sigma={SIGMA:0.##}");

// Tableau du payoff : nul sous le strike, lineaire au-dela
Console.WriteLine();
Console.WriteLine("   S_T   |  payoff max(S_T - K, 0)");
Console.WriteLine("---------+-------------------------");
for (double st = 90.0; st <= 110.0 + 1e-9; st += 5.0)
{
    double payoff = Math.Max(st - K, 0.0);
    Console.WriteLine($"{st,6:0.0}   |  {payoff,8:0.0}");
}
Console.WriteLine();
Console.WriteLine("Zone du payoff positif : S_T > " + K + " — le temps (valeur temps) fait le reste du prix.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Contrat : call europeen  S0=100  K=100  T=1 an  r=0.05  sigma=0.2


   S_T   |  payoff max(S_T - K, 0)


---------+-------------------------


  90.0   |       0.0


  95.0   |       0.0


 100.0   |       0.0


 105.0   |       5.0


 110.0   |      10.0


Zone du payoff positif : S_T > 100 — le temps (valeur temps) fait le reste du prix.


Le payoff est nul sous le strike : tout le prix du call vient de la probabilité risque-neutre de finir dans la monnaie, pondérée par le temps valeur. Les trois moteurs estiment **exactement la même quantité** — c'est ce qui rend leur confrontation loyale.

## 2. Black-Scholes : la formule fermée — étalon et plafond

Sous les hypothèses du modèle (GBM, $\sigma$ constante, taux constant, pas de dividende, hedging continu, exercice européen), le prix s'écrit en forme fermée :

$$C = S_0\,N(d_1) - K e^{-rT} N(d_2), \qquad d_{1,2} = \frac{\ln(S_0/K) + (r \pm \tfrac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}$$

La fonction normale $N(\cdot)$ est **implémentée depuis le bas** — par l'approximation d' Abramowitz-Stegun (7.1.26) de la fonction d'erreur — sans aucun package. Le coût de calcul est **une évaluation de fonction normale** : négligeable, indépendant de toute discrétisation.

In [2]:
// Fonction normale N(x), implementee depuis le bas (erf, Abramowitz-Stegun 7.1.26, erreur max 1.5e-7)
double NormalPdf(double x)
{
    return Math.Exp(-0.5 * x * x) / Math.Sqrt(2.0 * Math.PI);
}
double Erf(double x)
{
    double t = 1.0 / (1.0 + 0.3275911 * Math.Abs(x));
    double poly = ((((1.061405429 * t - 1.453152027) * t + 1.421413741) * t
                    - 0.284496736) * t + 0.254829592) * t;
    double e = 1.0 - Math.Exp(-x * x) * poly;
    return (x < 0.0) ? -e : e;
}
double NormalCdf(double x)
{
    return 0.5 * (1.0 + Erf(x / Math.Sqrt(2.0)));
}
double BsPrice(double S, double Kk, double Tt, double r, double sigma, string typ = "call")
{
    double d1 = (Math.Log(S / Kk) + (r + 0.5 * sigma * sigma) * Tt) / (sigma * Math.Sqrt(Tt));
    double d2 = d1 - sigma * Math.Sqrt(Tt);
    if (typ == "call")
        return S * NormalCdf(d1) - Kk * Math.Exp(-r * Tt) * NormalCdf(d2);
    return Kk * Math.Exp(-r * Tt) * NormalCdf(-d2) - S * NormalCdf(-d1);
}

double call_bs = BsPrice(S0, K, T, R, SIGMA, "call");
double put_bs = BsPrice(S0, K, T, R, SIGMA, "put");
double parity_rhs = S0 - K * Math.Exp(-R * T);
double parity_resid = call_bs - put_bs - parity_rhs;
Console.WriteLine($"Call europeen  (forme fermee)  : {call_bs:0.000000}");
Console.WriteLine($"Put europeen   (forme fermee)  : {put_bs:0.000000}");
Console.WriteLine($"Parite put-call  C - P - (S0 - K e^(-rT))  = {parity_resid:0.0000000}   (residu machine)");
Console.WriteLine($"RHS de la parite S0 - K e^(-rT)  = {parity_rhs:0.000000}");

Call europeen  (forme fermee)  : 10,450575


Put europeen   (forme fermee)  : 5,573518


Parite put-call  C - P - (S0 - K e^(-rT))  = -0,0000000   (residu machine)


RHS de la parite S0 - K e^(-rT)  = 4,877058


In [3]:
(double Delta, double Gamma, double Vega, double Theta) BsGreeks(double S, double Kk, double Tt, double r, double sigma, string typ = "call")
{
    double d1 = (Math.Log(S / Kk) + (r + 0.5 * sigma * sigma) * Tt) / (sigma * Math.Sqrt(Tt));
    double d2 = d1 - sigma * Math.Sqrt(Tt);
    double pdf = NormalPdf(d1);
    double delta = (typ == "call") ? NormalCdf(d1) : NormalCdf(d1) - 1.0;
    double gamma = pdf / (S * sigma * Math.Sqrt(Tt));
    double vega = S * pdf * Math.Sqrt(Tt) / 100.0;   // par point de vol (1 %)
    double theta = -S * pdf * sigma / (2.0 * Math.Sqrt(Tt));
    theta += (typ == "call")
        ? (-r * Kk * Math.Exp(-r * Tt) * NormalCdf(d2))
        : ( r * Kk * Math.Exp(-r * Tt) * NormalCdf(-d2));
    return (delta, gamma, vega, theta);
}
var g = BsGreeks(S0, K, T, R, SIGMA, "call");
Console.WriteLine("Greeks du call (forme fermee, a S0) :");
Console.WriteLine($"  delta = {g.Delta:0.0000}   gamma = {g.Gamma:0.000000}");
Console.WriteLine($"  vega  = {g.Vega:0.0000}  (par 1 pt de vol)   theta = {g.Theta:0.0000}  / an");

Greeks du call (forme fermee, a S0) :


  delta = 0,6368   gamma = 0,018762


  vega  = 0,3752  (par 1 pt de vol)   theta = -6,4140  / an


Deux enseignements déjà mesurables :

* **la formule est auto-cohérente** : la parité put-call est vérifiée au résidu machine — c'est le seul endroit du carnet où « exact » est un claim numérique littéral ;
* **le put vaut moins que le call** malgré un payoff symétrique en apparence : l'asymétrie vient de l'actualisation du strike et de la dérive risque-neutre positive.

Mais le plafond apparaît tout de suite : rien dans cette formule ne sait dire ce que vaut un **put américain** (exercable à tout instant), une option à **moyenne**, ou un payoff à **barrière**. La forme fermée est un étalon **pour le contrat qu'elle couvre** — pas un moteur général.

## 3. L'arbre binomial : convergence mesurée vers l'étalon

Le modèle Cox-Ross-Rubinstein (CRR) discrétise $T$ en $N$ pas de durée $\Delta t = T/N$ ; le sous-jacent monte à $u = e^{\sigma\sqrt{\Delta t}}$ ou descend à $d = 1/u$ avec probabilité risque-neutre $p = (e^{r\Delta t} - d)/(u-d)$. Le prix s'obtient par **induction descendante** depuis les payoffs terminaux.

La théorie promet une convergence en $O(1/N)$. La mesure montre une subtilité que tout praticien rencontre : l'erreur **oscille** autour de la limite au lieu de décroître monotone — l'arbre « voit » ou non le strike selon la parité de $N$.

In [4]:
double CrrPrice(double S, double Kk, double Tt, double r, double sigma, int N, string typ = "call", bool american = false)
{
    double dt = Tt / N;
    double u = Math.Exp(sigma * Math.Sqrt(dt));
    double d = 1.0 / u;
    double p = (Math.Exp(r * dt) - d) / (u - d);
    double disc = Math.Exp(-r * dt);
    double[] val = new double[N + 1];
    for (int j = 0; j <= N; j++)
    {
        double ST = S * Math.Pow(u, j) * Math.Pow(d, N - j);
        val[j] = (typ == "call") ? Math.Max(ST - Kk, 0.0) : Math.Max(Kk - ST, 0.0);
    }
    for (int i = N - 1; i >= 0; i--)
    {
        for (int j = 0; j <= i; j++)
        {
            double cont = disc * (p * val[j + 1] + (1.0 - p) * val[j]);
            double node = cont;
            if (american)
            {
                double Si = S * Math.Pow(u, j) * Math.Pow(d, i - j);
                double exo = (typ == "call") ? Math.Max(Si - Kk, 0.0) : Math.Max(Kk - Si, 0.0);
                node = Math.Max(cont, exo);
            }
            val[j] = node;
        }
    }
    return val[0];
}

// Convergence oscillante vers l'etalon Black-Scholes
Console.WriteLine("     N   |   prix CRR   |   |erreur| vs BS");
Console.WriteLine("---------+--------------+------------------");
int[] Ns = { 10, 20, 50, 100, 200, 500, 1000, 2000 };
foreach (int N in Ns)
{
    double px = CrrPrice(S0, K, T, R, SIGMA, N, "call");
    double err = Math.Abs(px - call_bs);
    Console.WriteLine($"{N,6}   |   {px:0.000000}   |   {err:0.000000}");
}

     N   |   prix CRR   |   |erreur| vs BS


---------+--------------+------------------


    10   |   10,253409   |   0,197166


    20   |   10,351260   |   0,099315


    50   |   10,410692   |   0,039884


   100   |   10,430612   |   0,019964


   200   |   10,440591   |   0,009984


   500   |   10,446585   |   0,003990


  1000   |   10,448584   |   0,001991


  2000   |   10,449584   |   0,000992


La mesure confirme les deux faits et les nuance :

* **l'ordre de grandeur est bien $O(1/N)$** — mais l'erreur locale **rebondit** : doubler $N$ peut temporairement *dégrader* la précision (un pair d'échelons voisins tombe des deux côtés de la limite). Un choix naïf de $N$ peut être moins précis qu'un choix plus petit bien placé ;
* **le coût total croît plus vite que linéairement en $N$** : la couche $i$ de l'induction traite $i$ nœuds, soit $N^2/2$ opérations au total. Diviser l'erreur par 2 coûte donc progressivement ~4× le calcul : le **même ordre asymptotique** que Monte-Carlo, mais une constante radicalement plus petite.

## 4. Le put américain : là où la formule fermée se tait

Le contrat change : même paramétrage, mais **put** et **exercice anticipé autorisé**. La valeur américaine est un problème d'arrêt optimal — à chaque nœud, le détenteur compare la continuation (valeur espérée) à l'exercice immédiat $K - S$.

Black-Scholes ne fournit **aucune réponse** à ce contrat, à aucune précision : ce n'est pas un manque de rigueur numérique, c'est une limite **structurelle** de la forme fermée. L'arbre, lui, traite l'arrêt optimal nativement : un `max(contenu, exercice)` à chaque nœud.

In [5]:
// Put europeen (reference fermee) vs put americain (arbre CRR)
double p_euro_bs = put_bs;
double p_euro_arbre = CrrPrice(S0, K, T, R, SIGMA, 1600, "put");
double p_us_arbre = CrrPrice(S0, K, T, R, SIGMA, 1600, "put", american: true);
double prime_anticipee = p_us_arbre - p_euro_arbre;
double c_us_arbre = CrrPrice(S0, K, T, R, SIGMA, 1600, "call", american: true);

Console.WriteLine($"Put europeen   (forme fermee BS) : {p_euro_bs:0.000000}");
Console.WriteLine($"Put europeen   (arbre, N=1600)   : {p_euro_arbre:0.000000}   (ecart: {Math.Abs(p_euro_arbre - p_euro_bs):0.000000})");
Console.WriteLine($"Put americain  (arbre, N=1600)   : {p_us_arbre:0.000000}");
Console.WriteLine($"=> prime d'exercice anticipee   : {prime_anticipee:0.000000}  ({prime_anticipee / p_euro_arbre:0.00%} du put europeen)");
Console.WriteLine();
Console.WriteLine($"Call americain (arbre) = {c_us_arbre:0.000000}   vs call europeen {call_bs:0.000000}  (Merton : ecart = erreur de discretisation)");

Put europeen   (forme fermee BS) : 5,573518


Put europeen   (arbre, N=1600)   : 5,572276   (ecart: 0,001242)


Put americain  (arbre, N=1600)   : 6,089893


=> prime d'exercice anticipee   : 0,517616  (9,29% du put europeen)


Call americain (arbre) = 10,449334   vs call europeen 10,450575  (Merton : ecart = erreur de discretisation)


Trois résultats mesurés :

1. **la prime d'exercice anticipé existe et est positive** pour le put — l'exercice immédiat encaisse $K$ plus tôt et le taux rémunère cette attente ;
2. **l'écart arbre/étalon sur le put européen** reste de l'ordre de la précision attendue pour $N$=1600 ;
3. **la vérification de Merton** (call américain = call européen sans dividende) tombe gratuitement : l'arbre ne trouve **jamais** d'exercice optimal.

C'est la première leçon de flexibilité : **l'arbre sait faire quelque chose que la formule ne sait pas**, à coût comparable.

## 5. Monte-Carlo : payer la précision au racine carré

Troisième moteur : simuler $M$ trajectoires terminales du GBM sous la probabilité risque-neutre,
$$S_T^{(m)} = S_0 \exp\!\Big[(r-\tfrac{\sigma^2}{2})T + \sigma\sqrt{T}\,Z_m\Big], \qquad Z_m \sim \mathcal{N}(0,1),$$
et estimer l'espérance actualisée par la moyenne empirique des payoffs. Le théorème central limite promet une erreur en $\sigma_{\text{payoff}}/\sqrt{M}$ avec un intervalle de confiance à 95 % — la mesure vérifie et surtout **rend sensible le régime de coût** : chaque chiffre de précision supplémentaire coûte ×100 en trajectoires.

In [6]:
double BoxMuller(System.Random rng)
{
    double u1 = 1.0 - rng.NextDouble();
    double u2 = 1.0 - rng.NextDouble();
    return Math.Sqrt(-2.0 * Math.Log(u1)) * Math.Cos(2.0 * Math.PI * u2);
}
(double Prix, double DemieIc) McPrice(double S, double Kk, double Tt, double r, double sigma, int M, int seed, bool antithetic = false)
{
    var rng = new System.Random(seed);
    double drift = (r - 0.5 * sigma * sigma) * Tt;
    double vol = sigma * Math.Sqrt(Tt);
    double disc = Math.Exp(-r * Tt);
    int P = antithetic ? M / 2 : M;
    double[] pay = new double[P];
    for (int m = 0; m < P; m++)
    {
        double z = BoxMuller(rng);
        double st1 = S * Math.Exp(drift + vol * z);
        double v = disc * Math.Max(st1 - Kk, 0.0);
        if (antithetic)
        {
            double st2 = S * Math.Exp(drift - vol * z);
            v = 0.5 * (v + disc * Math.Max(st2 - Kk, 0.0));
        }
        pay[m] = v;
    }
    double mean = 0.0;
    foreach (double v in pay) mean += v;
    mean /= P;
    double var = 0.0;
    foreach (double v in pay) var += (v - mean) * (v - mean);
    double sd = Math.Sqrt(var / (P - 1));
    return (mean, 1.96 * sd / Math.Sqrt(P));
}

// Convergence en 1/sqrt(M) : l'IC se resserre lentement
Console.WriteLine("       M   |   prix MC   |   demi-IC 95%");
Console.WriteLine("----------+--------------+----------------");
int[] Ms = { 1000, 10000, 100000, 1000000 };
foreach (int M in Ms)
{
    var res = McPrice(S0, K, T, R, SIGMA, M, seed: 42);
    Console.WriteLine($"{M,9}   |   {res.Prix:0.000000}   |   {res.DemieIc:0.000000}");
}
var ant = McPrice(S0, K, T, R, SIGMA, 100000, seed: 42, antithetic: true);
Console.WriteLine();
Console.WriteLine($"Antithetique M=100000 : {ant.Prix:0.000000}  (demi-IC {ant.DemieIc:0.000000})  — reduction de variance par paires (Z, -Z)");

       M   |   prix MC   |   demi-IC 95%


----------+--------------+----------------


     1000   |   10,603112   |   0,889852


    10000   |   10,594232   |   0,286958


   100000   |   10,444136   |   0,090981


  1000000   |   10,454572   |   0,028818


Antithetique M=100000 : 10,458472  (demi-IC 0,063977)  — reduction de variance par paires (Z, -Z)


## 6. Le régime de coût : le même ordre, des constantes lointaines

La lecture des trois sections se condense en une phrase : **à précision fixée, la formule est gratuite, l'arbre est nettement moins cher que la simulation, et la simulation est le seul moteur général**.

| Moteur | Ordre de convergence | Coût à précision fixée |
|---|---|---|
| Black-Scholes | exacte | 1 évaluation de $N(\cdot)$ |
| Arbre CRR | $O(1/N)$, oscillant | $\sim N^2$ opérations |
| Monte-Carlo | $O(1/\sqrt{M})$, IC | $\sim M$ trajectoires |

La simulation paie en $\sqrt{M}$ : un chiffre de précision supplémentaire coûte **×100** en trajectoires. C'est le prix de la généralité — le seul moteur à pricer un payoff qui n'a pas de forme fermée.

## 7. Quand la forme fermée s'arrête : le cas asiatique

Le call **asiatique arithmétique** (moyenne des $n$ spots) n'a **pas** de forme fermée pour sa moyenne arithmétique : la référence de long terme reste un **estimateur** (IC divulgué), pas une formule. C'est précisément le cas où la simulation n'est plus un choix de confort mais **le** moteur — et où les trois enseignements ci-dessus (IC, coût, généralité) deviennent la règle plutôt que l'exception.

> Le jumeau Python développe ce cas en détail (variable de contrôle par le payoff géométrique, dont la forme fermée est connue). Cette version C# s'arrête au paradigme : *quand une forme fermée n'existe pas pour la cible, la simulation porte la référence*.

## 8. Brancher ces pricers dans QuantConnect

Ce carnet tourne en local, mais ses moteurs se branchent directement dans l'écosystème QuantConnect du cours :

* **Recherche (QuantBook)** : les fonctions `BsPrice`, `CrrPrice`, `McPrice` se copient telles quelles dans un notebook de recherche QC pour pricer sur des données réelles ;
* **Backtest** : LEAN embarque des `OptionPriceModels` (Black-Scholes et binomial CRR intégrés) pour les filtres de contrats et le marking ; un pricer maison s'injecte comme modèle custom dans l'algorithme — c'est le pont vers [QC-Py-06](../Python/QC-Py-06-Options-Trading.ipynb) ;
* **Sizing et risque** : les greeks mesurés en section 2 pilotent le dimensionnement des positions.

## 9. Limites et garde-fous

Chaque moteur a été mesuré sur ses forces ; honnêteté oblige à mesurer aussi ce qu'aucun des trois ne dit :

* **la volatilité n'est pas constante** : tout ce carnet prix avec $\sigma = 20\%$ plat ; le marché affiche un smile/skew ;
* **le modèle, pas le marché** : le prix est exact *sous le modèle* (GBM, taux constants, pas de sauts, hedging continu) — ni le risque de gap, ni la liquidité, ni les coûts de transaction ne sont dans le prix ;
* **la précision mesurée est une précision numérique**, pas une garantie sur la valeur de marché.

## 10. Exercices

Quatre exercices pour s'approprier les moteurs. Chacun est stubbé : le carnet s'exécute de bout en bout, à vous d'écrire le corps (chaque stub retourne `null` sans erreur).

### Exercice 1 — La parité put-call à l'épreuve des trois moteurs

La parité $C - P = S_0 - K e^{-rT}$ est exacte en forme fermée. Les moteurs numériques la respectent-ils à leur précision respective ?

1. Calculer le put européen par (a) la formule fermée, (b) l'arbre à $N$=1600, (c) Monte-Carlo à $M$=256 000 ;
2. pour chaque moteur, former le résidu de parité $\hat{C} - \hat{P} - (S_0 - K e^{-rT})$ ;
3. comparer chaque résidu à la demi-largeur d'IC du moteur MC.

In [7]:
// Exercice 1 : parite put-call croisee sur les trois moteurs
// Etape 1 : prix du put par les trois moteurs (CrrPrice(..., "put"), McPrice adapte au put).
// Etape 2 : former les trois residus de parite.
// Etape 3 : comparer au demi-IC du MC.
double[] ResidusParite()
{
    // TODO etudiant
    return null;   // renvoyer le tableau {residu par moteur}
}
Console.WriteLine("Exercice 1 a completer : ResidusParite() retourne null (stub compilable).");

Exercice 1 a completer : ResidusParite() retourne null (stub compilable).


### Exercice 2 — Dividende discret : l'exercice anticipé du call devient optimal

Sans dividende, le call américain égal le call européen (Merton, vérifié en section 4). Introduisez un dividende discret $D$ payé à $t_d = 0{,}5T$ :

1. modifier le pricer d'arbre pour décrémenter le sous-jacent de $D$ après $t_d$ ;
2. mesurer la prime d'exercice anticipé du call pour $D \in \{0, 2, 5, 8\}$ ;
3. expliquer le signe et le seuil : pourquoi au-delà d'un certain dividende l'exercice juste avant $t_d$ domine ?

In [8]:
// Exercice 2 : call americain avec dividende discret
// Etape 1 : etendre CrrPrice avec un dividende (D, t_d) — arbre en deux segments.
// Etape 2 : prime d'exercice anticipee = prix americain - prix europeen (meme arbre).
// Etape 3 : trouver le D ou la prime devient strictement positive.
double CrrDividende(double S, double Kk, double Tt, double r, double sigma, int N, double D, double t_d, bool american)
{
    // TODO etudiant
    return 0.0;   // renvoyer le prix
}
Console.WriteLine("Exercice 2 a completer : CrrDividende() retourne 0.0 (stub compilable).");

Exercice 2 a completer : CrrDividende() retourne 0.0 (stub compilable).


### Exercice 3 — L'IC à 95 % ment-il ? Couverture empirique

La demi-largeur $1{,}96\,\hat{\sigma}/\sqrt{M}$ est une promesse statistique : sur 100 runs indépendants, environ 95 doivent couvrir la vraie valeur.

1. Fixer $M$=4 000 ; répéter $R$=300 runs **indépendants** (graines différentes) du MC du call ;
2. compter la fraction de runs dont l'IC contient l'étalon fermé ;
3. comparer à la couverture nominale 95 %.

In [9]:
// Exercice 3 : couverture empirique des IC
// Etape 1 : boucle de R runs a M fixe, graines distinctes (new Random(1000 + i)).
// Etape 2 : fraction_couverture = mean(|prix_i - call_bs| <= demie_ic_i).
// Etape 3 : comparer |fraction - 0.95| a la demi-largeur binomiale sqrt(0.95*0.05/R).
(double Fraction, double DemiBinomiale) CouvertureEmpirique(int M = 4000, int R = 300)
{
    // TODO etudiant
    return (0.0, 0.0);   // renvoyer (fraction_couverture, demi_largeur_binomiale)
}
Console.WriteLine("Exercice 3 a completer : CouvertureEmpirique() retourne (0.0, 0.0) (stub compilable).");

Exercice 3 a completer : CouvertureEmpirique() retourne (0.0, 0.0) (stub compilable).


### Exercice 4 — Le delta par différences finies : trois moteurs, une dérivée

Le delta analytique de la section 2 vaut $N(d_1)$. Chaque moteur numérique peut l'estimer par différence finie centrale : $\Delta \approx [P(S_0+h) - P(S_0-h)] / (2h)$.

1. implémenter le delta fini ($h$=1e-2 puis 1e-4) pour l'arbre ($N$=1600) et le MC ($M$=256 000, **même graine** pour les deux spots) ;
2. mesurer l'écart de chaque estimateur au delta analytique ;
3. expliquer pourquoi, pour le MC, la différence finie sans graine commune est dominée par le bruit de rééchantillonnage.

In [10]:
// Exercice 4 : delta par differences finies dans les trois moteurs
// Etape 1 : delta_fini(pricer, h) = (pricer(S0+h) - pricer(S0-h)) / (2h) pour arbre et MC.
// Indice MC : reutiliser EXACTEMENT la meme graine pour que seuls les spots different.
// Etape 2 : ecart absolu a BsGreeks(...).Delta pour h=1e-2 et h=1e-4.
double DeltaFini(double h)
{
    // TODO etudiant
    return 0.0;   // renvoyer l'estimation du delta
}
Console.WriteLine("Exercice 4 a completer : DeltaFini() retourne 0.0 (stub compilable).");

Exercice 4 a completer : DeltaFini() retourne 0.0 (stub compilable).


***

## Ce qu'il faut retenir

| | Forme fermée | Arbre binomial | Monte-Carlo |
|---|---|---|---|
| **Précision (contrat couvert)** | exacte | $O(1/N)$, oscillante | $O(1/\sqrt{M})$, IC |
| **Coût à précision fixée** | nul | $\sim N^2$ ops pour $1/N$ | $\sim M$ pour $1/\sqrt{M}$ |
| **Contrats couverts** | européen vanille | + américain (arrêt optimal) | tout payoff simulable |
| **Piège propre** | muet hors de ses hypothèses | oscillations de convergence | IC mal compris (exercice 3) |

1. **Trois moteurs, un seul contrat, des mesures** : le choix d'un pricer est une décision économique justifiée par des chiffres (précision, coût, flexibilité), pas une habitude.
2. **La flexibilité est la vraie ligne de fracture** : l'étalon fermé est gratuit mais aveugle ; l'arbre ouvre l'exercice anticipé ; la simulation ouvre les payoffs chemin-dépendants — et paie en racine carrée.

> Jumeau C# du [carnet Python QC-Py-06b](../Python/QC-Py-06b-Derivatives-Valuation-From-Scratch.ipynb) — mêmes nombres, même language de finance, language .NET. BCL pur, zéro package.